In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from tqdm import tqdm
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_time = os.path.join(path, 'Q1_data.csv')
df_time = pd.read_csv(data_time)

In [ ]:
# Task 2: Write your code here:
df_time.head()

In [ ]:
# Task 3: Write your code here:
df_time.info()

In [ ]:
# Task 4: Write your code here:
df_time.describe()

In [ ]:
# Task 5: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(df_time['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_time.drop(columns=["Order_ID"])


In [ ]:
# Task 2: Write your code here:
print(df_time.isnull().sum())
df_clean = df_time.dropna(subset = ["Weather","Traffic_Level","Time_of_Day","Courier_Experience_yrs"])

df_clean["Delivery_Time"] = df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mean())
print("-"*30)
print(df_clean.isnull().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)
df_clean.duplicated().sum()

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
print("-"*30)

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean.info()

In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()
features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET


df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
import seaborn as sns



def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")
#not balanced

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop(columns=["Delivery_Time"]).astype(float)
y = df_clean["Delivery_Time"].astype(float)


In [ ]:
# Task 2,3,4,5: Write your code here:

n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_mae = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model = RandomForestRegressor(n_estimators=200)
  model.fit(X_train,y_train)
  # Validate
  y_pred = model.predict(X_test)

  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)


  # Store results
  lr_mae.append(mae)

print()
print(f"  Average MAE: {np.mean(lr_mae):.4f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols = X

importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
lr_mae.plt.hist(bins=30, edgecolor='black')

plt.show()

In [ ]:
# Task Bonus: Write your code here: